# Credit Risk Intelligence — Fast Full 307,511-Row Pipeline (Google Colab)

Runs the complete unified platform on the full Home Credit dataset (`307,511` training rows + `48,744` test rows + all child history tables) using Colab's fast local SSD (`/content/spark_work`) to avoid slow Google Drive FUSE I/O:
1. Provision & verify OpenJDK 17/21 and set a valid `JAVA_HOME`.
2. Mount Google Drive and locate your repository + `_data/` folder.
3. Install Python & Spark/Delta dependencies (`pyspark==3.5.3`, `delta-spark==3.2.1`, `lightgbm`, `shap`).
4. Run quick Spark + Delta Lake smoke verification.
5. Stage `_data/` to local SSD and run the full unified pipeline (Bronze -> Silver -> Gold + Streaming + Real-Time Model + Gold LightGBM/LR/Cost-Opt/SHAP).
6. Download `credit_risk_full_artifacts.zip` (also saved directly into your Drive repo folder) and extract it locally.

In [8]:
#@title 1. Provision Java (JDK 17 preferred, fallback to pre-installed JDK 21) & verify JAVA_HOME
!rm -f /etc/apt/sources.list.d/*cran* /etc/apt/sources.list.d/*r2u* 2>/dev/null || true
!apt-get update -qq || true
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null || true

import os, shutil
from pathlib import Path

java_home = None
for candidate in (
    '/usr/lib/jvm/java-17-openjdk-amd64',
    '/usr/lib/jvm/java-17-openjdk',
    '/usr/lib/jvm/java-21-openjdk-amd64',
    '/usr/lib/jvm/default-java',
):
    if (Path(candidate) / 'bin' / 'java').exists():
        java_home = candidate
        break
if not java_home and shutil.which('java'):
    java_home = str(Path(shutil.which('java')).resolve().parents[1])

assert java_home and (Path(java_home) / 'bin' / 'java').exists(), 'No working Java runtime found!'
os.environ['JAVA_HOME'] = java_home
os.environ['PATH'] = f"{java_home}/bin:" + os.environ.get('PATH', '')
print('Verified JAVA_HOME =', os.environ['JAVA_HOME'])
!"$JAVA_HOME/bin/java" -version
!python --version

Verified JAVA_HOME = /usr/lib/jvm/java-17-openjdk-amd64
openjdk version "17.0.21.0.3" 2026-10-24
OpenJDK Runtime Environment (build 17.0.21.0.3+1-1-24.04.1-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.21.0.3+1-1-24.04.1-Ubuntu, mixed mode, sharing)
Python 3.13.18


In [9]:
#@title 2. Mount Google Drive and locate synced repository + _data/
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

CANDIDATE_DIRS = [
    '/content/drive/MyDrive/_hust/IT5427 - Big Data',
    '/content/drive/MyDrive/_hust/IT5427 - Big Data/it5427-credit_risk_bigdata',
    '/content/drive/MyDrive/it5427-credit_risk_bigdata',
]
REPO_DIR = next(
    (p for p in CANDIDATE_DIRS if (Path(p) / 'run_track_a.py').exists() and (Path(p) / '_data' / 'application_train.csv').exists()),
    None,
)
assert REPO_DIR is not None, f'Could not find repo + _data/application_train.csv in: {CANDIDATE_DIRS}'
os.chdir(REPO_DIR)
print('Working repo:', os.getcwd())
!ls -lh _data/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working repo: /content/drive/MyDrive/_hust/IT5427 - Big Data
total 2.5G
-rw------- 1 root root  26M Dec 10  2019 application_test.csv
-rw------- 1 root root 159M Dec 10  2019 application_train.csv
-rw------- 1 root root 359M Dec 10  2019 bureau_balance.csv
-rw------- 1 root root 163M Dec 10  2019 bureau.csv
-rw------- 1 root root 405M Dec 10  2019 credit_card_balance.csv
-rw------- 1 root root  37K Dec 10  2019 HomeCredit_columns_description.csv
-rw------- 1 root root 690M Dec 10  2019 installments_payments.csv
-rw------- 1 root root 375M Dec 10  2019 POS_CASH_balance.csv
-rw------- 1 root root 387M Dec 10  2019 previous_application.csv
-rw------- 1 root root 524K Dec 10  2019 sample_submission.csv


In [10]:
#@title 3. Install Python dependencies (ignore benign dataproc-spark-connect warning)
!pip install -q -r requirements.txt -r apache-spark/requirements.txt
!python -c "import pyspark, delta, lightgbm, shap; print('Deps OK | PySpark:', pyspark.__version__)"

Deps OK | PySpark: 3.5.3


In [11]:
#@title 4. Quick Spark & Delta Lake connectivity check (~10s)
%cd apache-spark
!CRB_TEST_DELTA=1 python -m pytest tests/test_06_delta_smoke.py -v
# Optional full 41-test Spark suite (adds a few minutes before pipeline start):
# !CRB_TEST_DELTA=1 python -m pytest tests/ -v
%cd ..

/content/drive/MyDrive/_hust/IT5427 - Big Data/apache-spark
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content/drive/MyDrive/_hust/IT5427 - Big Data/apache-spark
configfile: pytest.ini
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 2 items                                                              

tests/test_06_delta_smoke.py ..                                          [100%]

============================== 2 passed in 45.13s ==============================
/content/drive/MyDrive/_hust/IT5427 - Big Data


In [12]:
#@title 5. Run Unified Full 307,511-Row Pipeline (staged on local SSD for 5-10x speedup)
# Remove --fast if you also want to re-run pandas EDA + Spark column-level raw profiling.
!python scripts/run_colab_full_pipeline.py \
  --raw-dir _data \
  --work-dir /content/spark_work \
  --stage-raw \
  --fast \
  --resume-if-gold-exists \
  --driver-memory auto \
  --shuffle-partitions 32 \
  --verify-delta \
  --bundle credit_risk_full_artifacts.zip

Environment: JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64 | driver_memory=6g
Staging HomeCredit_columns_description.csv (0.0 MB) -> /content/spark_work/_data ...
Staging POS_CASH_balance.csv (392.7 MB) -> /content/spark_work/_data ...
Staging application_test.csv (26.6 MB) -> /content/spark_work/_data ...
Staging application_train.csv (166.1 MB) -> /content/spark_work/_data ...
Staging bureau.csv (170.0 MB) -> /content/spark_work/_data ...
Staging bureau_balance.csv (375.6 MB) -> /content/spark_work/_data ...
Staging credit_card_balance.csv (424.6 MB) -> /content/spark_work/_data ...
Staging installments_payments.csv (723.1 MB) -> /content/spark_work/_data ...
Staging previous_application.csv (405.0 MB) -> /content/spark_work/_data ...
Staging sample_submission.csv (0.5 MB) -> /content/spark_work/_data ...

>>> Running (/content/drive/MyDrive/_hust/IT5427 - Big Data): /usr/bin/python3 ingestion-minio-kafka/ingestion/gen_macro.py
region_dim.csv                             82 rows
region

In [ ]:
#@title 6. Download generated artifact bundle (also saved inside Drive REPO_DIR)
from google.colab import files
files.download('credit_risk_full_artifacts.zip')